# AssemLens • HP ZGX Nano overnight training starter
**Qwen3-VL-4B + BF16 LoRA • Assembly101 • 22 September 2026**

Tonight's deliverable is a real trained **assembly-action recognition adapter**, saved checkpoints, and a baseline-versus-adapter validation report. The inputs are two chronological images of an annotated action; the output is action/verb/object JSON. This is a domain-adaptation experiment, **not yet a furniture mistake detector**. Improvement is measured, not guaranteed.

The language attention adapters train; the base weights and vision encoder stay frozen. Two frames are a cheap starting point and can miss short actions. We deliberately avoid quantization and custom FlashAttention builds for this first Arm/Blackwell run.

**Run on the remote HP, not your Mac.** Your screenshot identifies a ZGX Nano, but this notebook checks the actual CUDA device rather than assuming its memory. Do not replace the HP's vendor PyTorch with the old desktop `cu128` command. MLflow installation is not required.

Run sections in order. Downloads require internet and dataset access. Default budget: up to 4 train + 2 validation recordings from one camera, 80 action segments per recording, 2 epochs, and a 6-hour training-loop limit. Downloads, baseline evaluation and final evaluation are outside that limit. This may finish sooner than overnight. Reserve **at least 35 GiB free disk** for model, videos, frames and checkpoints.

**Validation status:** notebook/worker syntax and data-selection invariants checked by the author; no HP GPU, gated dataset download, or end-to-end model training was available in the authoring environment. The two-step pilot below is mandatory before leaving the machine.


## 1. Open the notebook on the HP and preserve its working CUDA environment
Your VS Code status bar already shows SSH. In that remote window, create/open `~/assemlens`, then upload this notebook into that folder (drag into the **remote Explorer**, or use SCP). Install/enable the Python and Jupyter extensions **on SSH: hp15** if prompted.

In the remote terminal, use `conda env list` to locate the Python environment installed by HP. Activate that environment using its actual name, then check it:
```bash
python -c "import torch; print(torch.__version__, torch.cuda.is_available()); print(torch.cuda.get_device_name(0))"
```
If this fails, fix/select the HP CUDA environment before continuing. A Mac Python interpreter cannot train this job.

From that verified environment, create an isolated environment that inherits its vendor GPU packages. Run these commands in the **remote terminal**, not notebook `!source` cells:
```bash
mkdir -p ~/assemlens
cd ~/assemlens
python -m venv --system-site-packages .venv
source .venv/bin/activate
python -c "import importlib.metadata as m; print('\n'.join(n+'=='+m.version(n) for n in ['torch','torchvision']))" > hp-torch-constraints.txt
uv pip install --python .venv/bin/python -c hp-torch-constraints.txt \
  'transformers==4.57.1' 'peft==0.17.1' 'accelerate==1.10.1' \
  'huggingface-hub>=0.34,<1' 'pillow>=11,<13' 'pandas>=2.2,<3' ipykernel
python -m ipykernel install --user --name assemlens-hp --display-name 'Python (AssemLens HP)'
```
If `uv` is unavailable, substitute `python -m pip install` for `uv pip install --python .venv/bin/python`. If torchvision is not installed in the vendor environment, omit it from the constraint-generation list; this notebook uses PIL/FFmpeg. If dependencies conflict, stop and inspect the conflict—do not upgrade the GPU stack blindly.

This notebook also needs **FFmpeg**. Check `ffmpeg -version`. If absent, install it in an isolated Conda tools environment:
```bash
conda create -y -n assemlens-media -c conda-forge ffmpeg
```
Then set `FFMPEG` in the configuration cell to the executable in that environment, e.g. the path reported by `conda run -n assemlens-media which ffmpeg`.

Select notebook **kernel → Python Environments → `~/assemlens/.venv/bin/python`**, or the registered **Python (AssemLens HP)** kernel. Terminal activation alone does not change the notebook kernel.


In [ ]:
import sys, platform, shutil, subprocess, json, os, time
from pathlib import Path
import torch
print('Python:', sys.executable)
print('Host:', platform.node(), '| architecture:', platform.machine())
print('Torch:', torch.__version__, '| CUDA runtime:', torch.version.cuda)
assert platform.system() == 'Linux', 'Select the remote HP Linux kernel, not your Mac.'
assert torch.cuda.is_available(), 'Select the HP CUDA-enabled Python environment.'
print('GPU:', torch.cuda.get_device_name(0))
print('Reported device memory GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1))
assert torch.cuda.is_bf16_supported(), 'This run requires BF16 support.'
x = torch.randn(512, 512, device='cuda', dtype=torch.bfloat16)
y = x @ x
assert torch.isfinite(y).all()
del x, y
torch.cuda.empty_cache()
print('CUDA matrix multiplication passed.')


## 2. Configure one small, reproducible experiment
Use a **new ROOT directory** when changing the subset, frames, model or training settings. Existing checkpoints automatically resume only when the saved signature matches. The first run does not need a camera.


In [ ]:
ROOT = (Path.home() / 'assemlens' / 'runs' / 'assembly101_v1').resolve()
ROOT.mkdir(parents=True, exist_ok=True)
FFMPEG = shutil.which('ffmpeg')  # Or paste absolute path from assemlens-media environment.
assert FFMPEG and Path(FFMPEG).is_file(), 'Install FFmpeg as described above, then set FFMPEG.'
CFG = dict(root=str(ROOT), model='Qwen/Qwen3-VL-4B-Instruct',
           dataset='cvml-nus/assembly101', camera='C10404_rgb.mp4',
           train_recordings=4, validation_recordings=2, segments_per_recording=80,
           video_budget_gib=8, image_side=448, training_hours=6)
assert shutil.disk_usage(ROOT).free > 35 * 2**30, 'Free at least 35 GiB before starting.'
subprocess.run([FFMPEG, '-version'], check=True, stdout=subprocess.DEVNULL)
print(json.dumps(CFG, indent=2))


## 3. Accept dataset conditions and authenticate
Open [Assembly101 on Hugging Face](https://huggingface.co/datasets/cvml-nus/assembly101), sign in, and accept its access conditions. It is **CC BY-NC 4.0**, with attribution and noncommercial restrictions; it is not unrestricted commercial training data. Read the conditions applicable to your event/use. Do not download the entire repository (approximately 3.89 TB at research time).

Create a Hugging Face **read token** with access to this dataset. This cell prompts securely and caches it locally; never paste tokens into notebook source or share your credential cache. If access is denied, resolve the account/access issue before running overnight. No bypass or unofficial rehost is used.


In [ ]:
from huggingface_hub import HfApi, hf_hub_download, login, get_token
from getpass import getpass
if not get_token():
    login(token=getpass('Hugging Face read token: '), add_to_git_credential=False)
api = HfApi()
CFG['dataset_revision'] = api.dataset_info(CFG['dataset']).sha
CFG['model_revision'] = api.model_info(CFG['model']).sha
# A small authorized download proves access before downloading any videos.
actions_path = hf_hub_download(CFG['dataset'], 'annotations/fine-grained-annotations/actions.csv',
                               repo_type='dataset', revision=CFG['dataset_revision'])
print('Access verified. Dataset revision:', CFG['dataset_revision'])
(ROOT / 'config.json').write_text(json.dumps(CFG, indent=2))
# Freeze installed package versions for the experiment; no credentials in this command.
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## 4. Select official train/validation recordings and inspect download budget
Use only one RGB camera. Keep complete recording identities separate across splits to prevent adjacent-frame leakage. The untouched official test set is not used tonight. These are small **validation** results, not a benchmark leaderboard claim.

The official fine-grained annotation frame numbers are at **30 fps**, even though raw recordings can be 60 fps. We convert annotations to seconds using 30, then decode by timestamp. Select deterministic recordings and samples; this tiny subset is not class balanced.


In [ ]:
import pandas as pd
import random
columns = ['id','video','start_frame','end_frame','action_cls','verb_cls','noun_cls']
tables = {}
for split in ['train','validation']:
    path = hf_hub_download(CFG['dataset'], f'annotations/fine-grained-annotations/{split}.csv',
                           repo_type='dataset', revision=CFG['dataset_revision'])
    df = pd.read_csv(path, usecols=columns).dropna()
    df = df[df.video.str.endswith('/'+CFG['camera'])].copy()
    df['sequence'] = df.video.str.split('/').str[0]
    duration = (df.end_frame - df.start_frame)/30.0
    df = df[(duration >= 1.0) & (duration <= 30.0)].copy()
    tables[split] = df
# Exclude all official validation recording identities from the training pool.
val_sequences = set(tables['validation'].sequence)
tables['train'] = tables['train'][~tables['train'].sequence.isin(val_sequences)].copy()
selected = {}
for split in ['train','validation']:
    df = tables[split]
    sequences = sorted(df.sequence.unique())
    random.Random(42).shuffle(sequences)
    sequences = sequences[:CFG[split+'_recordings']]
    assert len(sequences) >= (2 if split == 'train' else 1), f'Too few {split} recordings.'
    selected[split] = pd.concat([df[df.sequence == seq].sample(n=min(CFG['segments_per_recording'], len(df[df.sequence == seq])), random_state=42) for seq in sequences]).sort_values(['video','start_frame'])
assert not (set(selected['train'].sequence) & set(selected['validation'].sequence))
video_plan=[]
for video in sorted(set(pd.concat(list(selected.values())).video)):
    sequence, camera = video.split('/')
    repo_path = 'recordings/'+video
    entries = list(api.list_repo_tree(CFG['dataset'], path_in_repo='recordings/'+sequence,
                                     repo_type='dataset', revision=CFG['dataset_revision']))
    matches=[e for e in entries if e.path == repo_path]
    assert len(matches)==1 and matches[0].size > 0, f'Missing video metadata: {repo_path}'
    video_plan.append(dict(video=video, path=repo_path, bytes=matches[0].size))
size=sum(v['bytes'] for v in video_plan)
print('Video download GiB:',round(size/2**30,2))
print({s:dict(recordings=int(d.sequence.nunique()), segments=len(d)) for s,d in selected.items()})
assert size <= CFG['video_budget_gib']*2**30, 'Budget exceeded: reduce recording counts and rerun selection. No videos downloaded yet.'
(ROOT/'video_plan.json').write_text(json.dumps(video_plan,indent=2))
for split, df in selected.items(): df.to_csv(ROOT/f'{split}_selection.csv',index=False)


## 5. Download only selected videos and extract two frames per action
Downloads can take longer than training on a slow connection. Cached Hugging Face files are reused. This is the preparation stage, so keep the notebook connected until it finishes. Images remain private local training artifacts; don't publish dataset videos without considering its license.


In [ ]:
from PIL import Image, ImageOps, ImageDraw
import hashlib
videos={}
for i, item in enumerate(video_plan, 1):
    print(f'Downloading {i}/{len(video_plan)}:', item['video'], flush=True)
    videos[item['video']] = hf_hub_download(CFG['dataset'], item['path'], repo_type='dataset', revision=CFG['dataset_revision'])

def extract_frame(video, seconds, destination):
    destination=Path(destination)
    if destination.exists():
        with Image.open(destination) as im: im.verify()
        return
    temp=destination.with_suffix('.tmp.png')
    cmd=[FFMPEG,'-hide_banner','-loglevel','error','-y','-ss',f'{seconds:.4f}',
         '-i',str(video),'-frames:v','1',str(temp)]
    result=subprocess.run(cmd,capture_output=True,text=True,timeout=90)
    if result.returncode or not temp.exists():
        raise RuntimeError('Frame extraction failed: '+result.stderr[-500:])
    with Image.open(temp) as im:
        im=im.convert('RGB'); im.thumbnail((CFG['image_side'],CFG['image_side']))
        im.save(destination,quality=92)
    temp.unlink()

manifests={}; failures=[]
for split, df in selected.items():
    folder=ROOT/'frames'/split; folder.mkdir(parents=True,exist_ok=True)
    rows=[]
    for number, row in enumerate(df.itertuples(),1):
        key=hashlib.sha256(f'{split}:{row.video}:{row.id}:{row.start_frame}:{row.end_frame}'.encode()).hexdigest()[:20]
        start,end=row.start_frame/30.0,row.end_frame/30.0
        paths=[folder/f'{key}_{i}.jpg' for i in range(2)]
        try:
            for path, frac in zip(paths,[0.2,0.8]): extract_frame(videos[row.video],start+(end-start)*frac,path)
            rows.append(dict(id=key,sequence=row.sequence,video=row.video,start_seconds=start,end_seconds=end,
                             images=[str(p) for p in paths],target=dict(action=row.action_cls,verb=row.verb_cls,object=row.noun_cls)))
        except Exception as e:
            failures.append(dict(split=split,id=key,error=str(e)))
        if number%40==0: print(split,number,'/',len(df),flush=True)
    assert len(rows) >= 8, 'Too few decoded examples.'
    assert len(rows) >= 0.9*len(df), 'More than 10% failed to decode: inspect alignment/FFmpeg before training.'
    manifests[split]=rows
    (ROOT/f'{split}.jsonl').write_text(''.join(json.dumps(r)+'\n' for r in rows))
(ROOT/'decode_failures.json').write_text(json.dumps(failures,indent=2))
print('Prepared:',{s:len(r) for s,r in manifests.items()},'| decode failures:',len(failures))


## 6. Human sanity check — don't skip this
Inspect the first and last sampled frame against the label. They should show the same action segment, in chronological order. If frames are blank, labels don't match visible activity, or the camera gives unusable views, stop and fix the data before training. A caption label is not proof that a connection is correct or a screw is tight.


In [ ]:
from IPython.display import display
for row in manifests['train'][::max(1,len(manifests['train'])//6)][:6]:
    canvas=Image.new('RGB',(640,250),'white')
    for i,path in enumerate(row['images']):
        with Image.open(path) as im:
            tile=ImageOps.contain(im.convert('RGB'),(315,210)); canvas.paste(tile,(i*320,0))
    ImageDraw.Draw(canvas).text((5,218),row['target']['action'],fill='black')
    display(canvas)


## 7. Write the training worker
This cell creates a standalone local script. LoRA targets only the language model's attention projections. Prompt and image positions are masked out of the loss; only the assistant answer is supervised. Training uses microbatch 1, accumulation 8, rank 16, and SDPA.

The worker saves adapters, optimizer checkpoints, baseline/adapted predictions, validation loss, a package snapshot, and status. Exact-label matching is deliberately strict: synonyms count as errors, so inspect predictions too. With small samples, a lower score can be noise or overfitting; don't advertise improvement unless supported.


In [ ]:
# Load the version-controlled worker instead of duplicating its code here.
candidates = [Path.cwd(), Path.cwd().parent]
REPO = next((p for p in candidates if (p / 'scripts/train_assemlens.py').is_file()), None)
assert REPO is not None, 'Open this notebook from the cloned assemlens repository.'
WORKER = ROOT / 'train_assemlens.py'
WORKER.write_text((REPO / 'scripts/train_assemlens.py').read_text())
print(WORKER)


## 8. Run the mandatory pilot (two optimizer steps)
This loads the model, computes a tiny baseline, trains, saves the adapter, then evaluates. It may take several minutes on the first download. Read the log if it fails. The pilot uses a separate output directory; the full job starts from the base weights.

If CUDA runs out of memory: close other GPU workloads, reduce `image_side` to 336 in a **new ROOT**, rebuild frames, and rerun. Do not run two training workers at once. If imports/CUDA fail, fix the environment rather than starting the overnight job.


In [ ]:
pilot_log=ROOT/'pilot.log'
with pilot_log.open('w') as log:
    result=subprocess.run([sys.executable,'-u',str(WORKER),'--config',str(ROOT/'config.json'),'--pilot'],stdout=log,stderr=subprocess.STDOUT)
print(pilot_log.read_text()[-10000:])
assert result.returncode == 0, f'Pilot failed. Inspect {pilot_log}'
assert json.loads((ROOT/'pilot/status.json').read_text())['state']=='finished'
print('Pilot passed. Ready for the full job.')


## 9. Start the bounded overnight job
Run once after the pilot succeeds. It detaches from the notebook session, logs to disk, and saves a checkpoint every 25 optimizer steps. You can disconnect VS Code after launch, but **the HP must stay powered on** and your lab must permit persistent processes. Closing the Mac lid won't preserve the job if the lab's session policy kills user jobs.

A restart resumes the latest checkpoint in this run directory. Up to the last 25 steps may need repeating after an interruption. The 6-hour training timer resets on resume. A completed run is protected from accidental relaunch.


In [ ]:
status_path=ROOT/'overnight/status.json'
if status_path.exists():
    old=json.loads(status_path.read_text())
    assert old.get('state')!='finished', 'This run is complete. Inspect results or choose a new ROOT.'
    pid=old.get('pid')
    if pid:
        try:
            os.kill(pid,0)
        except ProcessLookupError:
            pass
        else:
            raise RuntimeError(f'PID {pid} still exists; check it before launching another worker.')
assert json.loads((ROOT/'pilot/status.json').read_text())['state']=='finished', 'Run the pilot first.'
with (ROOT/'overnight.log').open('a') as log:
    process=subprocess.Popen([sys.executable,'-u',str(WORKER),'--config',str(ROOT/'config.json')],
                             stdin=subprocess.DEVNULL,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
(ROOT/'worker_pid.txt').write_text(str(process.pid))
print('Launched PID:',process.pid)
print('Log:',ROOT/'overnight.log')
print('Wait a minute, then run the monitor cell. Do not click launch again.')


In [ ]:
# Rerun this cell whenever you want an update; it does not launch training.
if (ROOT/'overnight/status.json').exists(): print((ROOT/'overnight/status.json').read_text())
log=ROOT/'overnight.log'
if log.exists():
    with log.open('rb') as f:
        f.seek(max(0,log.stat().st_size-12000)); print(f.read().decode(errors='replace'))


## 10. Read tomorrow's results
`overnight/adapter/` is the deliverable adapter, not a standalone full model: retain the exact base-model revision in `config.json`. Keep the whole run folder for reproducibility. No paid inference API is used; compute, storage and dataset terms still apply.

Report baseline and adapted action/verb/object exact match on the same validation samples, sample count, inference latency, GPU hardware, and training duration from logs. This experiment uses known annotated action boundaries; it does **not** measure continuous-video action segmentation or real-time performance. Do not use these validation clips for final-demo test claims after tuning repeatedly on them.


In [ ]:
comparison=ROOT/'overnight/comparison.json'
if comparison.exists():
    report=json.loads(comparison.read_text()); print(json.dumps(report,indent=2))
    display(pd.DataFrame({'baseline':report['baseline'],'adapted':report['adapted']}))
else:
    print('No final report yet. Check status/log; a checkpoint alone does not mean evaluation finished.')


## 11. What to record for a credible AssemLens demo
**Must you use the same furniture? No. But transfer is an experiment, not a guarantee.** This dataset contains toy vehicles, so this adapter has not learned your office chair, its manual, or its failure states. For tonight's output, show a held-out toy assembly clip and the model's action predictions, clearly labeled as the action-recognition stage.

For the full product demonstration, choose **one** accessible assembly and 4–6 visible states. Best fit to this dataset: a take-apart toy vehicle with large pieces. A simple tabletop organizer or small flat-pack item is also easier to film than a whole chair. Use something you already own if its parts can be safely reassembled. No purchase is necessary for tonight's run.

Your office chair is usable if its manual and visible assembly stages are available: e.g. an armrest on the wrong side, a visibly reversed bracket, or an omitted visible part. Do not dismantle or manipulate the pressurized gas cylinder for the demo. Do not claim camera-only verification of torque, hidden fasteners or load-bearing safety.

**Tomorrow, build the actual error-verification training set:**
1. Record 12–20 short sessions of your selected object with correct state, deliberate visible mistake, correction, and obstructed/uncertain views. Film several people, lighting conditions and angles.
2. Keep full sessions separate: roughly 70% train, 15% validation, 15% final test. Never randomly split adjacent frames across these sets.
3. Annotate each example with the step, required reference state, observed state, error location if visible, short correction, and one of complete/incomplete/incorrect/uncertain. Include correct examples so the model learns when to stay quiet. Have a second teammate review ambiguous labels.
4. Adapt the worker's prompt/targets to that schema and include the product reference image/criteria as inputs. Start a **separate experiment**, comparing the pretrained model against both this warm-up and direct target-task adaptation; this warm-up may not help.
5. Drive your existing AssemLens state machine with the verified output. Require repeat observations before advancing; an action such as “attach” is not enough to confirm success.

**Recorded demo storyboard (60–90 seconds):** QR selects the supported product → clip shows assembly → visible mistake → concise correction overlay → corrected view → explicit verified state. Use predictions from your actual model run, allow abstention, and keep a held-out recording. If analysis is precomputed or sped up, label it and report actual latency. You do not need a live camera connected to the HP for this.

This notebook intentionally does not pretend the action adapter can already produce trustworthy corrections. The custom-label stage is necessary before making that claim.

## Sources and attribution
- [Assembly101 official dataset and access/license](https://huggingface.co/datasets/cvml-nus/assembly101): Sener et al., Assembly101, CVPR 2022. This run extracts/resizes frames and samples a subset; original labels remain human annotations.
- [Official fine-grained annotation schema and 30 fps convention](https://github.com/assembly-101/assembly101-annotations/tree/main/fine-grained-annotations).
- [Official Assembly101 mistake-detection task](https://github.com/assembly-101/assembly101-mistake-detection): a distinct supervision task, not consumed by this notebook.
- [Qwen3-VL official repository](https://github.com/QwenLM/Qwen3-VL) and [4B Instruct model card](https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct).
- [PEFT LoRA documentation](https://huggingface.co/docs/peft/en/package_reference/lora).


## 12. Event requirements — updated from your six kickoff handout photos
**Submission: Friday, September 25, 2026 by 8 p.m.** The handout does not specify a time zone; use the organizer's event-local deadline and verify the submission portal. Set an internal cutoff of 6 p.m. Your primary inference must run on the assigned ZGX Nano. The handout calls this an **Edge AI systems contest**, not a model demo contest. Fine-tuning is encouraged, not stated as a mandatory minimum training workload.

**Missing from this training notebook, but necessary in your application:** an explicit, defensible, measurable cloud-escalation policy. This notebook does not implement a cloud service or satisfy the full contest by itself.

Suggested product positioning: AssemLens helps a first-time flat-pack furniture owner assemble one supported product, while keeping workspace video local. The Nano is the local inference server; the Mac browser is the client. Do not claim inference happens on the Mac. A consumer product would need access to a suitable local server or a different deployment plan.

**Proposed routing policy to implement and test:**
- Clear, consistent evidence and a supported product: evaluate locally; the state machine advances only after repeated independent observations.
- Occlusion/blur: ask for a clearer angle locally. A larger cloud model does not fix absent evidence.
- Persistent disagreement after two clear observations, or a manual question outside the local reference package: consider cloud escalation only if the user permits it, internet is available, and a request/time budget allows it. Log the reason and what data left the device. Prefer a selected crop or text question to a continuous video feed.
- No permission, no internet, or exhausted budget: abstain, preserve current step, and request another view or human/manual help. Do not mark complete by default.

These are proposed starting rules, not calibrated thresholds. Choose thresholds on validation sessions; test on separate sessions. Don't treat a VLM's self-reported confidence as a calibrated probability. Demonstrate the cloud-disabled path and, if implemented, one real escalation. Label a mocked service as mocked; it is not evidence of cloud quality.

**System measurements:** error-detection precision/recall; false-completion rate; false interventions; local p50/p95 end-to-end latency; local resolution/abstention/escalation rates; cloud latency, payload size and cost per session; behavior when WAN access is unavailable. Compare the same held-out scenarios with the base model, the fine-tuned model and the routing policy. If you lack a cloud-only measurement, do not invent one.

**Event-safe development:** coordinate GPU slots among all five teammates; don't run serving and training concurrently unless measured and budgeted. Use only your assigned node. Download the bounded subset sequentially and coordinate network-heavy work; don't saturate shared egress or use multi-node examples. Test an offline application flag that blocks cloud calls rather than disabling your remote machine's network and losing SSH. Don't reboot the node and walk away.

**ZTK / ZRT / vLLM:** ZTK manages remote development; ZRT is the handout's vLLM serving wrapper; this notebook uses PyTorch/PEFT for training. They are different stages. You can first serve the adapter with Transformers in your app on the Nano. If moving to ZRT/vLLM, verify the installed version supports your model and multimodal LoRA, or explicitly merge/export and validate output parity. Do not assume a PEFT adapter folder is a full vLLM model. The handout's Qwen2.5-VL-7B medical fine-tuning example is a useful hardware-specific reference, but copying its medical data doesn't help your assembly task. NVFP4 serving is optional for this starter.

**Security detail in the example to change:** don't copy `echo $HF_TOKEN` into a recorded terminal, public notebook or logs. Do not broadly disable proxy authentication/TLS or bind an unauthenticated service to all interfaces just to fix a port error. Keep a service on loopback and forward the app port through VS Code/SSH for the demo. Use separate ports for model API and UI. Keep credentials and private captures out of your public repo.

**Deliverables checklist:**
- Public GitHub repository with code, README, a Dockerfile **or setup script**, dependency pins, model revision, annotation schema and reproduction commands. Include metrics and explain why you chose them.
- Save the adapter/checkpoints somewhere durable off the Nano before the event ends; nodes may be wiped. Use an appropriate model/artifact store or approved backup; reference large artifacts from the repo instead of committing raw datasets or multi-GB weights. Respect dataset license/access restrictions.
- Working demo on the Nano through SSH/tunneled port. The handout allows a recorded demonstration when pitch/infrastructure subnets prevent a live demo; prepare a 5-minute walkthrough and confirm the recording arrangement with organizers.
- Separate **maximum 2-minute video**, public YouTube URL **and video file**.
- SJSU Drive package: project brief, pitch video, links, presentation, diagrams and photos.
- Accessible, dynamic final presentation: one architecture visual, one benchmark/evidence visual, one impact visual; prepare for 3 minutes of Q&A.
- Socials are listed in the handout; use the exact tags there. Publishing is your team's action, not done by this notebook.

**Revised priorities:** September 22: environment + pilot + bounded experiment + record target object; September 23: target-task labels, correction loop and routing; September 24: held-out tests, outage behavior and demo recording; September 25: reproduce on Nano, back up artifacts, finalize video/README and submit before 8 p.m. Do not trade away integration time for more generic-data epochs.
